# Lab 2: Regular Expressions, Edit Distance, and Subword Unit Tokenization

In this lab, we will cover key topics, including regular expressions, edit distance, and subword unit tokenization using Byte Pair Encoding (BPE). This notebook contains exercises that students will need to complete in the code sections. The necessary files will also be provided with this notebook.


## Exercise 1: Basic Regular Expressions

Write regular expressions for the following languages:
1. The set of all alphabetic strings
2. The set of all lower-case alphabetic strings ending in the letter b
3. The set of all strings from the alphabet a,b such that each letter a is immediately preceded by and immediately followed by the letter b

In [1]:
import re

# 1. The set of all alphabetic strings
regex_1 = r'^[A-Za-z]+$'

# 2. The set of all lower-case alphabetic strings ending in the letter b
regex_2 = r'^[a-z]*b$'

# 3. The set of all strings from the alphabet a,b such that each letter a is immediately preceded by and immediately followed by the letter b
regex_3 = r'^b*(bab)*b*$'

# Test the regular expressions
test_strings = ["", 'Hello', 'world', 'abc', 'bab', 'abab', 'baba', 'babbabbab', 'bababab', 'abba']

for i, (regex, question) in enumerate([(regex_1, "all alphabetic strings"),
                                        (regex_2, "all lower-case alphabetic strings ending in a b"),
                                        (regex_3, "all strings from the alphabet a,b such that each a is immediately preceded by and immediately followed by a b")], 1):
    print(f"\nTesting regex {regex} which searches for {question}:")
    for test_str in test_strings:
        print(f'"{test_str}" matches: {bool(re.match(regex, test_str))}')


Testing regex ^[A-Za-z]+$ which searches for all alphabetic strings:
"" matches: False
"Hello" matches: True
"world" matches: True
"abc" matches: True
"bab" matches: True
"abab" matches: True
"baba" matches: True
"babbabbab" matches: True
"bababab" matches: True
"abba" matches: True

Testing regex ^[a-z]*b$ which searches for all lower-case alphabetic strings ending in a b:
"" matches: False
"Hello" matches: False
"world" matches: False
"abc" matches: False
"bab" matches: True
"abab" matches: True
"baba" matches: False
"babbabbab" matches: True
"bababab" matches: True
"abba" matches: False

Testing regex ^b*(bab)*b*$ which searches for all strings from the alphabet a,b such that each a is immediately preceded by and immediately followed by a b:
"" matches: True
"Hello" matches: False
"world" matches: False
"abc" matches: False
"bab" matches: True
"abab" matches: False
"baba" matches: False
"babbabbab" matches: True
"bababab" matches: False
"abba" matches: False


## Exercise 2: Advanced Regular Expressions

Write regular expressions for the following languages. By "word", we mean an alphabetic string separated from other words by whitespace, any relevant punctuation, line breaks, and so forth.

1. The set of all strings with two consecutive repeated words (e.g., "Humbert Humbert" and "the the" but not "the bug" or "the big bug").
2. All strings that start at the beginning of the line with an integer and that end at the end of the line with a word.
3. All strings that have both the word grotto and the word raven in them (but not, e.g., words like grottos that merely contain the word grotto).
4. Write a pattern that captures the first word of an English sentence along with any ending punctuation if it is concatenated to it.

In [2]:
import re

# 1. Two consecutive repeated words
regex_1 = r'\b(\w+)\s+\1\b'

# 2. Starts with integer, ends with word
regex_2 = r'^\d+\b.*\b\w+$'

# 3. Contains both 'grotto' and 'raven' as whole words
regex_3 = r'\b(grotto|raven)\b.*\b(grotto|raven)\b'

# 4. First word of an English sentence
regex_4 = r'^\W*(\w[\w\'-]*)[^\w\s]*\b'

# Test the regular expressions
test_strings = [
    "Humbert Humbert is a character character",
    "The big bug",
    "42 is the answer to everything",
    "The raven visited the grotto",
    "Hello, world! How are you?"
]

for i, regex in enumerate([regex_1, regex_2, regex_3, regex_4], 1):
    print(f"\nTesting regex {i}:")
    for test_str in test_strings:
        match = re.search(regex, test_str)
        if match:
            print(f'"{test_str}" matches: "{match.group()}"')
        else:
            print(f'"{test_str}" does not match')


Testing regex 1:
"Humbert Humbert is a character character" matches: "Humbert Humbert"
"The big bug" does not match
"42 is the answer to everything" does not match
"The raven visited the grotto" does not match
"Hello, world! How are you?" does not match

Testing regex 2:
"Humbert Humbert is a character character" does not match
"The big bug" does not match
"42 is the answer to everything" matches: "42 is the answer to everything"
"The raven visited the grotto" does not match
"Hello, world! How are you?" does not match

Testing regex 3:
"Humbert Humbert is a character character" does not match
"The big bug" does not match
"42 is the answer to everything" does not match
"The raven visited the grotto" matches: "raven visited the grotto"
"Hello, world! How are you?" does not match

Testing regex 4:
"Humbert Humbert is a character character" matches: "Humbert"
"The big bug" matches: "The"
"42 is the answer to everything" matches: "42"
"The raven visited the grotto" matches: "The"
"Hello, w

## Exercise 3: Regular Expression for Employee Data Extraction

Using the provided HTML file containing employee information, write regular expressions to extract the following data: Full Name, Email Address, Phone Number, Personal Identification Code, Date of Birth, Website URL, City, Country, Job Title, Department.

Create a regex for each data point and save the extracted employee data in a JSON file.

In [18]:
import re
import json

# Read the HTML content from the file
html_content = open("employee-data-html.html", "r", encoding='utf-8').read()

# Define regular expressions for each data point
patterns = {
    'job_title': r'<h3>(.*?)</h3>',
    'first_name': r'<span class="firstname">(.*?)</span>',
    'last_name': r'<span class="lastname">(.*?)</span>',
    'email': r'Email:\s*<[^>]*>(.*?@.*?\..*?)<',
    'phone': r'Téléphone:\s*<[^>]*>\+?[\d\s\-()]+<',
    'identification_code': r'Code d\'identification:\s*<[^>]*>([A-Z]{2}\d{2}#[\w@!*]+)<',
    'date_of_birth': r'Date de naissance:\s*<[^>]*>(\d{4}-\d{2}-\d{2})<',
    'website': r'Site Web:\s*<[^>]*>(https?://|www\.)?([^\s<]+)<',
    'city': r'Ville:\s*<[^>]*>([^,<]+)',
    'department': r'<h2>(.*?)</h2>'
}

# Extract employees with their departments
employees = []
current_department = None

# Split HTML by employee cards
cards = re.findall(r'<div class="employee-card">(.*?)</div>', html_content, re.DOTALL)

# Also extract departments
departments = re.findall(r'<h2>(.*?)</h2>', html_content)

# Track department index
dept_index = 0
card_count = 0
cards_per_dept = [5, 3, 2, 2, 2]  # Number of employees per department

for card_idx, card in enumerate(cards):
    # Determine current department
    if dept_index < len(cards_per_dept) and card_count >= cards_per_dept[dept_index]:
        card_count = 0
        dept_index += 1
    
    if dept_index < len(departments):
        current_department = departments[dept_index]
    
    card_count += 1
    
    # Extract data from current card
    employee = {}
    
    # Job Title
    title_match = re.search(r'<h3>(.*?)</h3>', card)
    employee['job_title'] = title_match.group(1).strip() if title_match else None
    
    # First and Last Name
    fname_match = re.search(r'<span class="firstname">(.*?)</span>', card)
    lname_match = re.search(r'<span class="lastname">(.*?)</span>', card)
    employee['first_name'] = fname_match.group(1).strip() if fname_match else None
    employee['last_name'] = lname_match.group(1).strip() if lname_match else None
    employee['full_name'] = f"{employee['first_name']} {employee['last_name']}" if employee['first_name'] and employee['last_name'] else None
    
    # Email
    email_match = re.search(r'Email:\s*(?:<[^>]*>)?([\w.-]+@[\w.-]+\.\w+)', card)
    employee['email'] = email_match.group(1).strip() if email_match else None
    
    # Phone Number
    phone_match = re.search(r'Téléphone:\s*(?:<[^>]*)?\+?([\d\s\-()]+?)(?:<|$)', card)
    if phone_match:
        employee['phone'] = '+' + phone_match.group(1).strip()
    else:
        employee['phone'] = None
    
    # Identification Code
    id_match = re.search(r'Code d\'identification:\s*(?:<[^>]*)?([\w#@!*]+?)(?:<|$)', card)
    employee['identification_code'] = id_match.group(1).strip() if id_match else None
    
    # Date of Birth
    dob_match = re.search(r'Date de naissance:\s*(?:<[^>]*)?(\\d{4}-\\d{2}-\\d{2})(?:<|$)', card)
    if not dob_match:
        dob_match = re.search(r'(\d{4}-\d{2}-\d{2})', card)
    employee['date_of_birth'] = dob_match.group(1) if dob_match else None
    
    # Website URL
    website_match = re.search(r'Site Web:\s*(?:<[^>]*)?(https?://|www\.)?([^\s<]+)(?:<|$)', card)
    if website_match:
        url = website_match.group(0).replace('Site Web:', '').replace('<[^>]*>', '').strip()
        employee['website_url'] = url
    else:
        employee['website_url'] = None
    
    # City and Country
    city_match = re.search(r'Ville:\s*(.+?)</p>', card)
    if city_match:
        city_info = city_match.group(1).strip()
        parts = [p.strip() for p in city_info.split(',')]
        employee['city'] = parts[0] if parts else None
        employee['country'] = parts[1] if len(parts) > 1 else None
    else:
        employee['city'] = None
        employee['country'] = None
    
    # Department
    employee['department'] = current_department
    
    employees.append(employee)

# Save to JSON file
output_file = "employees.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(employees, f, ensure_ascii=False, indent=2)

print(f"✓ Successfully extracted {len(employees)} employees")
print(f"✓ Data saved to {output_file}")

# Display sample output
print("\n--- Sample Employee Record ---")
if employees:
    print(json.dumps(employees[0], ensure_ascii=False, indent=2))

✓ Successfully extracted 14 employees
✓ Data saved to employees.json

--- Sample Employee Record ---
{
  "job_title": "Ingénieur Logiciel",
  "first_name": "Karim",
  "last_name": "Bendaoud",
  "full_name": "Karim Bendaoud",
  "email": "k.bendaoud@techmaghreb.dz",
  "phone": "+213 551 23 45 67",
  "identification_code": "BD93#k4R1m*25",
  "date_of_birth": "1993-05-25",
  "website_url": "https://karim-bendaoud.dz<",
  "city": "Alger",
  "country": "Algérie",
  "department": "Département d'Ingénierie"
}


## Exercise 4: Create a Chatbot Inspired by ELIZA

Design a simple chatbot that mimics the style of ELIZA, an early AI program created in 1966 by Joseph Weizenbaum. ELIZA simulated a conversation with a psychotherapist by using pattern matching and simple responses.

In [9]:
import re
import random

class ElizaChatbot:
    def __init__(self):
        self.response_rules = [
            (r'hi|hello|hey', ["Hello! How are you feeling today?", "Hi there! What's on your mind?"]),
            (r'i am (.*)', ["Why do you say you are %1?", "How does being %1 make you feel?"]),
            (r'i feel (.*)', ["Why do you feel %1?", "Tell me more about feeling %1."]),
            (r'i (.*) you', ["Why do you %1 me?", "What makes you think I %1 you?"]),
            (r'you are (.*)', ["Why do you think I am %1?", "Does it please you to think I am %1?"]),
            (r'(.*) sorry (.*)', ["There's no need to apologize.", "Why do you feel the need to say sorry?"]),
            (r'(.*) mother (.*)|(.*) father (.*)', ["Tell me more about your family.", "How did your %2 make you feel?", "How did your %4 make you feel?"]),
            (r'(.*) help (.*)', ["I'm here to help. What would you like to talk about?", "How can I assist you with %2?"]),
            (r'(.*)', ["Please tell me more.", "I see. Can you elaborate?", "Why do you say that?"]),
        ]

    def eliza_response(self, user_input):
        for pattern, responses in self.response_rules:
            match = re.match(pattern, user_input, re.IGNORECASE)
            if match:
                # Substitute captured groups into the response
                response = random.choice(responses)
                # Only replace placeholders for groups that exist in the match
                for i in range(1, match.lastindex + 1 if match.lastindex else 1):
                    response = response.replace(f'%{i}', match.group(i) or '')
                return response
        return "I didn't quite understand that. Can you elaborate?"

# Example usage
if __name__ == "__main__":
    eliza = ElizaChatbot()
    print("ELIZA: Hello! I'm your virtual assistant. Type 'Quit' to exit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            print("ELIZA: Goodbye! It was nice talking to you.")
            break
        print("ELIZA:", eliza.eliza_response(user_input))

ELIZA: Hello! I'm your virtual assistant. Type 'Quit' to exit.
ELIZA: Hi there! What's on your mind?
ELIZA: Goodbye! It was nice talking to you.


## Exercise 5: Edit Distance

1. Compute the edit distance (using insertion cost 1, deletion cost 1, substitution cost 1) of "leda" to "deal". Show your work (using the edit distance grid).

2. Implement a minimum edit distance algorithm and use your hand-computed results to check your code.

In [11]:
from rich.console import Console
from rich.table import Table
from rich.text import Text

def edit_distance(s1: str, s2: str) -> tuple[int, list[list[int]]]:
    m: int = len(s1)
    n: int = len(s2)
    dp: list[list[int]] = [[0] * (n + 1) for _ in range(m + 1)]

    # Initialize the first row and column
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    # Fill the grid
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    return dp[m][n], dp


def print_edit_distance_grid(s1: str, s2: str, dp: list[list[int]]) -> None:
    console = Console()
    table = Table(title=f"Edit Distance Grid: '{s1}' → '{s2}'", show_lines=True)

    # Header row (target string)
    header_row = [" "] + [" "] + list(s2)
    for header in header_row:
        table.add_column(header, justify="center")

    # Fill the table
    for i in range(len(s1) + 1):
        row = []

        # Left column (source string)
        row.append(s1[i - 1] if i > 0 else " ")

        # Fill the row values
        for j in range(len(s2) + 1):
            cell_value = dp[i][j]
            if i > 0 and j > 0:
                if s1[i - 1] == s2[j - 1]:
                    text = Text(str(cell_value), style="bold green")
                else:
                    text = Text(str(cell_value), style="bold red")
            else:
                text = Text(str(cell_value), style="cyan")
            row.append(text)

        table.add_row(*row)

    console.print(table)


source: str = "leda"
target: str = "deal"

distance, dp = edit_distance(source, target)
print_edit_distance_grid(source, target, dp)

console = Console()
console.print(
    f"\n[bold yellow]The minimum edit distance from '{source}' to '{target}' is:[/bold yellow] "
    f"[bold cyan]{distance}[/bold cyan]\n"
)

   Edit Distance Grid:   
     'leda' → 'deal'     
┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃   ┃ d ┃ e ┃ a ┃ l ┃
┡━━━╇━━━╇━━━╇━━━╇━━━╇━━━┩
│   │ 0 │ 1 │ 2 │ 3 │ 4 │
├───┼───┼───┼───┼───┼───┤
│ l │ 1 │ 1 │ 2 │ 3 │ 3 │
├───┼───┼───┼───┼───┼───┤
│ e │ 2 │ 2 │ 1 │ 2 │ 3 │
├───┼───┼───┼───┼───┼───┤
│ d │ 3 │ 2 │ 2 │ 2 │ 3 │
├───┼───┼───┼───┼───┼───┤
│ a │ 4 │ 3 │ 3 │ 2 │ 3 │
└───┴───┴───┴───┴───┴───┘

The minimum edit distance from 'leda' to 'deal' is: 3

# Exercise 6: Spell Correction using NLTK

In this exercise, we'll implement spell correction functions using NLTK and the concepts of edit distance.

## Question 1:
Write a function that takes a misspelled word and returns the closest valid word from a given textual corpus using edit distance.

**Hint:** You can use the top 5000 most frequent words from an NLTK Gutenberg text such as 'austen-emma.txt'.

## Question 2:
Use the function you created in Question 1 to correct spelling errors in full sentences.

**Hint:** You can randomly introduce spelling errors into existing sentences from the Gutenberg Corpus available in NLTK by altering long words (those with more than 5 characters) through the insertion of a random character.


## Question 3:
Implement the following steps to evaluate the effectiveness of your spell correction:

1. **Select Sentences**: Choose a set of sentences from the Gutenberg Corpus available in NLTK.
2. **Introduce Spelling Errors**: Randomly alter some long words in each sentence to create spelling errors (through the insertion of a random character). Ensure that only words longer than 5 characters as before.
3. **Correct the Sentences**: Use the function from Question 1 to correct the misspelled words in each altered sentence.
4. **Print Results**: For each original sentence, print the altered sentence and the suggested corrections for the misspelled words.
5. **Calculate Precision**: Evaluate the accuracy of the corrections by calculating the precision of the spell correction method on the provided test set in the code section below.


In [13]:
import nltk
from nltk.corpus import gutenberg
from nltk import FreqDist
from nltk.tokenize import word_tokenize
from nltk.metrics import edit_distance
import random
import string
from sklearn.metrics import precision_score

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('gutenberg')
nltk.download('words')

# Define the vocab size
Vocab_size = 5000

# Load text and tokenize
tokens = [word.lower() for word in word_tokenize(gutenberg.raw('austen-emma.txt')) if word.isalpha()]

# Get top 5000 words
fdist = FreqDist(tokens)
vocab = [word for word, _ in fdist.most_common(Vocab_size)]

# Function to get closest word
def get_closest_word(misspelled_word, vocab):
    return min(vocab, key=lambda word: edit_distance(misspelled_word, word))

# Function to introduce errors
def introduce_errors(sentence):
    words = word_tokenize(sentence)
    altered_words = []
    for word in words:
        if len(word) > 5 and word.isalpha():
            random_char = random.choice(string.ascii_lowercase)
            pos = random.randint(0, len(word))
            altered_word = word[:pos] + random_char + word[pos:]
            altered_words.append(altered_word)
        else:
            altered_words.append(word)
    return ' '.join(altered_words)

# Function to correct sentence
def correct_sentence(sentence, vocab):
    words = word_tokenize(sentence)
    corrected_words = []
    for word in words:
        if word.lower() not in vocab:
            closest_word = get_closest_word(word.lower(), vocab)
            corrected_words.append(closest_word)
        else:
            corrected_words.append(word)
    return ' '.join(corrected_words)

# Evaluate spell correction
def evaluate_spell_correction(test_sentences, vocab):
    original_words = []
    altered_words = []
    corrected_words = []
    for sentence in test_sentences:
        altered_sentence = introduce_errors(sentence)
        corrected_sentence = correct_sentence(altered_sentence, vocab)
        original_words.extend(word_tokenize(sentence))
        altered_words.extend(word_tokenize(altered_sentence))
        corrected_words.extend(word_tokenize(corrected_sentence))

    y_true = [1 if orig == corr else 0 for orig, corr in zip(original_words, corrected_words)]
    y_pred = [1 if alt != corr else 0 for alt, corr in zip(altered_words, corrected_words)]
    precision = precision_score(y_true, y_pred)
    return precision

# Test sentences
test_sentences = [
    "She was pleased to have such a friend.",
    "He thought it was a great opportunity.",
    "The sun was shining brightly in the sky.",
    "They were walking towards the market together.",
    "I hope you will come to the party."
]

# Evaluate
precision = evaluate_spell_correction(test_sentences, vocab)
print(f"Precision: {precision:.2f}")

# Print results
for sentence in test_sentences:
    altered = introduce_errors(sentence)
    corrected = correct_sentence(altered, vocab)
    print(f"Original: {sentence}")
    print(f"Altered: {altered}")
    print(f"Corrected: {corrected}\n")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wwwab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\wwwab\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\wwwab\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


Precision: 0.53
Original: She was pleased to have such a friend.
Altered: She was plzeased to have such a friiend .
Corrected: She was pleased to have such a friend i

Original: He thought it was a great opportunity.
Altered: He thougqht it was a great opportunibty .
Corrected: He thought it was a great opportunity i

Original: The sun was shining brightly in the sky.
Altered: The sun was shiniang brightply in the sky .
Corrected: The sun was thinking rightly in the sky i

Original: They were walking towards the market together.
Altered: They were walkizng towardns the markket togetoher .
Corrected: They were walking towards the market together i

Original: I hope you will come to the party.
Altered: I hope you will come to the party .
Corrected: I hope you will come to the party i



# Exercise 7: Implementing Byte Pair Encoding (BPE) Tokenization

In this exercise, you will implement the Byte Pair Encoding (BPE) tokenization method step by step. Follow the instructions below to build your BPE tokenizer.

### Step 1: Get Vocabulary

Implement the function `get_vocab(sentences)` that takes a list of sentences and returns a dictionary with the counts of each token in the vocabulary.

### Step 2: Get Stats

Implement the function `get_stats(vocab)` that takes the vocabulary dictionary and returns a dictionary of pairs of tokens and their frequencies.

### Step 3: Merge Vocabulary

Implement the function `merge_vocab(vocab, pair)` that takes the vocabulary and a pair of tokens to merge. It should return the updated vocabulary.

### Step 4: BPE

Implement the function `bpe(sentences, num_merges)` that uses the previously defined functions to perform BPE tokenization on the provided sentences.

### Step 5: Apply BPE

Implement the function `apply_bpe(sentence, vocab)` that takes a sentence and the vocabulary and returns the tokenized version of the sentence using BPE.

Once you have completed all steps, test your implementation with sample sentences to observe how the BPE tokenization works. You can adjust the number of merges to see how it affects the tokenization.


In [14]:
from collections import defaultdict
import nltk
from nltk.tokenize import word_tokenize
from rich import print as rprint
from rich.panel import Panel
from rich.syntax import Syntax
from rich.table import Table

nltk.download('punkt')

# Step 1: Get Vocabulary
def get_vocab(sentences):
    vocab = defaultdict(int)
    for sentence in sentences:
        words = word_tokenize(sentence.lower())
        for word in words:
            tokens = list(word) + ['</w>']
            vocab[' '.join(tokens)] += 1
    return vocab

# Step 2: Get Stats
def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        tokens = word.split()
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pairs[pair] += freq
    return pairs

# Step 3: Merge Vocabulary
def merge_vocab(vocab, pair):
    new_vocab = defaultdict(int)
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab

# Step 4: BPE (Byte Pair Encoding)
def bpe(sentences, num_merges):
    vocab = get_vocab(sentences)
    merges = []
    for _ in range(num_merges):
        stats = get_stats(vocab)
        if not stats:
            break
        most_frequent_pair = max(stats, key=stats.get)
        merges.append(most_frequent_pair)
        vocab = merge_vocab(vocab, most_frequent_pair)
    return vocab, merges

# Step 5: Apply BPE
def apply_bpe(sentence, merges):
    words = word_tokenize(sentence.lower())
    tokenized_words = []
    for word in words:
        tokens = list(word) + ['</w>']
        for pair in merges:
            bigram = ' '.join(pair)
            replacement = ''.join(pair)
            i = 0
            while i < len(tokens) - 1:
                if tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                    tokens[i] = replacement
                    del tokens[i + 1]
                else:
                    i += 1
        if tokens and tokens[-1] == '</w>':
            tokens = tokens[:-1]
        tokenized_words.extend(tokens)
    return tokenized_words

# Rich CLI Printing
def print_results(vocab, merges, tokenized_sentence):
    # Print vocabulary
    table = Table(title="Vocabulary After BPE")
    table.add_column("Tokenized Word", style="cyan")
    table.add_column("Frequency", style="magenta")
    for word, freq in vocab.items():
        table.add_row(word, str(freq))
    rprint(table)

    # Print merges
    rprint(Panel.fit("[bold]Learned Merges:[/bold]"))
    for merge in merges:
        rprint(f"  - {merge[0]} + {merge[1]}")

    # Print tokenized sentence
    rprint(Panel.fit(f"[bold]Tokenized Sentence:[/bold] {tokenized_sentence}"))

# Example Usage
if __name__ == "__main__":
    sentences = [
        "Low",
        "lowest",
        "the",
        "this",
        "fastest"
    ]
    num_merges = 10
    vocab, merges = bpe(sentences, num_merges)
    sentence = "lowder in the"
    tokenized_sentence = apply_bpe(sentence, merges)
    print_results(vocab, merges, tokenized_sentence)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wwwab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


     Vocabulary After BPE     
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Tokenized Word ┃ Frequency ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ low</w>        │ 1         │
│ lowest</w>     │ 1         │
│ the</w>        │ 1         │
│ th i s </w>    │ 1         │
│ f a st est</w> │ 1         │
└────────────────┴───────────┘

╭─────────────────╮
│ Learned Merges: │
╰─────────────────╯

- s + t

- l + o

- lo + w

... (4 outputs left out)

- low + est</w>

- th + e

- the + </w>

╭─────────────────────────────────────────────────────────────────╮
│ Tokenized Sentence: ['low', 'd', 'e', 'r', 'i', 'n', 'the</w>'] │
╰─────────────────────────────────────────────────────────────────╯

# Exercise 8: Using SentencePiece for Byte Pair Encoding (BPE) Tokenization

In this exercise, you will learn how to use the SentencePiece library to implement Byte Pair Encoding (BPE) for tokenization. Follow the steps below to tokenize text using BPE.

### Step 1: Install SentencePiece

Ensure that you have the SentencePiece library installed in your Python environment. You can install it using pip:

```python
!pip install sentencepiece
```

### Step 2: Prepare Your Text
Load and merge all texts from the Gutenberg corpus available in NLTK. You can use the following code snippet to achieve this:

### Step 3: Train the BPE Model
Utilize SentencePiece to train a BPE model on the merged text. Specify the vocabulary size you want to achieve during the training process. For example:
```python
import sentencepiece as spm

# Train the SentencePiece model
spm.SentencePieceTrainer.Train('--input=merged_gutenberg.txt --model_prefix=m --vocab_size=5000')
```


### Step 4: Tokenize the Text
Once the model is trained, use it to tokenize your text into subword units. You can test the tokenizer with the following sample sentences:

```python
# Load the trained model
sp = spm.SentencePieceProcessor(model_file='m.model')

# Tokenize a sentence
tokens = sp.encode(sentence, out_type=str)
```

### Step 5: What about Arabic Texts !
Try to do the same thing but this time using an Arabic text corpus.

In [16]:
# Install SentencePiece
!pip install sentencepiece

# Import libraries
import nltk
from nltk.corpus import gutenberg
import sentencepiece as spm

# Download the Gutenberg corpus
nltk.download('gutenberg')

# Step 2: Prepare English Text
file_ids = gutenberg.fileids()
merged_text = ""
for file_id in file_ids:
    merged_text += gutenberg.raw(file_id) + "\n"

with open('merged_gutenberg.txt', 'w', encoding='utf-8') as f:
    f.write(merged_text)

# Step 3: Train BPE Model (English)
spm.SentencePieceTrainer.Train(
    input='merged_gutenberg.txt',
    model_prefix='gutenberg_bpe',
    vocab_size=5000,
    model_type='bpe',
    character_coverage=1.0,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

# Step 4: Tokenize English Text
sp = spm.SentencePieceProcessor(model_file='gutenberg_bpe.model')
sentence = "This is a sample sentence for BPE tokenization."
tokens = sp.encode(sentence, out_type=str)

print("Original English sentence:", sentence)
print("Tokenized:", tokens)

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\wwwab\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


Original English sentence: This is a sample sentence for BPE tokenization.
Tokenized: ['▁This', '▁is', '▁a', '▁sa', 'm', 'ple', '▁sent', 'ence', '▁for', '▁B', 'P', 'E', '▁to', 'ken', 'iz', 'ation', '.']


## Main differences between our simple BPE implementation and SentencePiece BPE:

### 1. End-of-word marker:
- **Simple BPE:** Uses `</w>` at the end of each word.
- **SentencePiece:** Uses `_` (underscore) at the beginning of each word.

**Example:**
- Simple BPE: `"low</w> est</w>"`
- SentencePiece: `"_low _est"`

---

### 2. Tokenization approach:
- **Simple BPE:** Tokenizes words first, then applies BPE.
- **SentencePiece:** Treats the entire input as a sequence of characters, including spaces.

**Example:**
- Simple BPE: `["low</w>", "est</w>"]`
- SentencePiece: `["", "l", "o", "w", "", "e", "s", "t"]`

---

### 3. Handling of whitespace:
- **Simple BPE:** Whitespace is removed during tokenization.
- **SentencePiece:** Preserves whitespace as part of the tokenization process.

**Example:**
- Simple BPE: `"hello world"` -> `["hello</w>", "world</w>"]`
- SentencePiece: `"hello world"` -> `["_hello", "_world"]`

---

### 4. Subword regularization:
- **Simple BPE:** No built-in regularization.
- **SentencePiece:** Supports subword regularization techniques like BPE-dropout.

---

### 5. Unicode handling:
- **Simple BPE:** Limited Unicode support (depends on NLTK tokenization).
- **SentencePiece:** Full Unicode support, treating each Unicode character as a basic unit.

---

### 6. Vocabulary size control:
- **Simple BPE:** Controlled by number of merge operations.
- **SentencePiece:** Directly specify desired vocabulary size.

---

### 7. Out-of-vocabulary (OOV) handling:
- **Simple BPE:** No specific OOV handling.
- **SentencePiece:** Has built-in OOV handling, often using a special `<unk>` token.

---

### 8. Reversibility:
- **Simple BPE:** Not easily reversible due to `</w>` placement.
- **SentencePiece:** Easily reversible due to `_` placement at word beginnings.

**Example:**
- SentencePiece: `"_app le"` -> `"apple"` (easily reversible)
- Simple BPE: `"app le</w>"` -> `"apple"` (requires special handling of `</w>`)

---

### 9. Efficiency and speed:
- **Simple BPE:** Generally very slow, especially for large datasets.
- **SentencePiece:** Highly optimized for speed and efficiency.
